# imports

In [1]:
%reload_ext autoreload
%autoreload 2

import torch
import sys
import os
# from torchmetrics.functional.pairwise import pairwise_cosine_similarity
import numpy as np
# from tqdm import tqdm
# from matplotlib import pyplot as plt
from datasets import load_dataset
import json
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification
)
import pyarrow.parquet as pq

ROOT_PATH = os.path.join(
    (os.path.dirname(os.path.abspath(''))),
)

# local imports
sys.path.insert(
    0,
    ROOT_PATH
)
# from evaluation.bright.retrievers import (
#     get_scores,
#     calculate_retrieval_metrics
# )
from utility.utils_for_notebooks import compute_doc_embeds
sys.path.pop(0)

DEBUG_NUMBER_EMBEDS = 100
DATASET_SOURCE = 'xlangai/BRIGHT'
CACHE_DIR = os.path.join(ROOT_PATH, "evaluation", "bright", "cache")
TASK = "theoremqa_theorems"

/home/oh/arubinstein17/github/ReasonIR/envs/reasonir/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# functions

In [2]:
def get_queries(examples):
    queries = []
    query_ids = []
    excluded_ids = {}
    separate_reasoning = False
    reasoning_examples = []
    for qid, e in enumerate(examples):
        if separate_reasoning:
            new_query = f"{e['query']}\n<REASON>\n{reasoning_examples[qid]['query']}"
            queries.append(new_query)
        else:
            queries.append(e["query"])
        query_ids.append(e['id'])
        excluded_ids[e['id']] = e['excluded_ids']
        overlap = set(e['excluded_ids']).intersection(set(e['gold_ids']))
        assert len(overlap)==0
    assert len(queries)==len(query_ids), f"{len(queries)}, {len(query_ids)}"
    return queries, query_ids


def check_augmented_docs(augmented_docs_path):
    # augmented_docs_path = os.path.join(
    #     ROOT_PATH,
    #     "output",
    #     "documents_augmentation_llama3_debug",
    #     "theoremqa_theorems_Llama-3.1-8B-Instruct_None.parquet"
    # )

    # Read the parquet file
    augmented_docs = pq.read_table(augmented_docs_path)
    augmented_docs = augmented_docs.to_pandas()

    # print(augmented_docs)

    doc_ids_augmented = []
    documents_augmented = []
    # for dp in augmented_docs:
    for _, dp in augmented_docs.iterrows():
        # print(dp)
        # print(dp['doc_id'])
        # print(dp['document'])
        doc_ids_augmented.append(str(dp['doc_id']))
        documents_augmented.append(dp['document'])

    print(len(documents_augmented))
    print(documents_augmented[0])
    print(doc_ids_augmented[:10])

# compute embeds for documents

## Augmented docs

In [4]:
os.listdir(os.path.join(ROOT_PATH, "output", "documents_augmentation_llama3"))

['theoremqa_theorems_Llama-3.1-8B-Instruct_None_10000:15000.parquet',
 'theoremqa_theorems_Llama-3.1-8B-Instruct_None_15000:20000.parquet',
 'theoremqa_theorems_Llama-3.1-8B-Instruct_None_0:5000.parquet',
 'theoremqa_theorems_Llama-3.1-8B-Instruct_None.parquet',
 'theoremqa_theorems_Llama-3.1-8B-Instruct_None_20000:25000.parquet',
 'theoremqa_theorems_Llama-3.1-8B-Instruct_None_5000:10000.parquet']

In [5]:
augmented_docs_path_prefix = os.path.join(
    ROOT_PATH,
    "output",
    "documents_augmentation_llama3"
)

# print(augmented_docs)
doc_ranges = [
    "0:5000",
    "5000:10000",
    "10000:15000",
    "15000:20000",
    "20000:25000"
]
# theoremqa_theorems_Llama-3.1-8B-Instruct_None_0:3.parquet
augmented_docs_path_list = [
    os.path.join(
        augmented_docs_path_prefix,
        f"theoremqa_theorems_Llama-3.1-8B-Instruct_None_{doc_range}.parquet"
    )
        for doc_range in doc_ranges
]
# augmented_docs_path_list = [
#     os.path.join(augmented_docs_path_prefix, path)
#         for path in augmented_docs_path_list
# ]

doc_ids_augmented = []
documents_augmented = []
# for dp in augmented_docs:
for augmented_docs_path in augmented_docs_path_list:
    # Read the parquet file
    augmented_docs = pq.read_table(augmented_docs_path)
    augmented_docs = augmented_docs.to_pandas()
    for _, dp in augmented_docs.iterrows():
        # print(dp)
        # print(dp['doc_id'])
        # print(dp['document'])
        doc_ids_augmented.append(str(dp['doc_id']))
        documents_augmented.append(dp['document'])

In [8]:
print(len(documents_augmented))
print(documents_augmented[-1])
print(doc_ids_augmented[-1])

23839
\section{Non-Archimedean Division Ring Iff Non-Archimedean Completion}
Tags: Complete Metric Spaces, Definitions: Norm Theory, Division Rings, Normed Division Rings, Definitions: Division Rings, Norm Theory, Definitions: Complete Metric Spaces, Non-Archimedean Norms

\begin{theorem}
Let $\struct {R, \norm {\, \cdot \,} }$ be a normed division ring.
Let $\struct {R', \norm {\, \cdot \,}' }$ be a normed division ring completion of $\struct {R, \norm {\, \cdot \,} }$
Then:
:$\norm {\, \cdot \,}$ is non-archimedean {{iff}} $\norm {\, \cdot \,}'$ is non-archimedean.
\end{theorem}

\begin{proof}
By the definition of a normed division ring completion then:
:$(1): \quad$ there exists a distance-preserving ring monomorphism $\phi: R \to R'$.
:$(2): \quad \struct {R', \norm {\, \cdot \,}' }$ is a complete metric space.
:$(3): \quad \phi \sqbrk R$ is a dense subspace in $\struct {R', \norm {\, \cdot \,}' }$.
By Normed Division Ring is Dense Subring of Completion, $\phi \sqbrk R$ is a dense 

In [9]:
doc_emb = compute_doc_embeds(
    documents=documents_augmented,
    task="theoremqa_theorems",
    save_path="/home/oh/arubinstein17/github/ReasonIR/evaluation/bright/theoremqa_theorems/augmented_doc_emb.pkl"
)

Loading checkpoint shards: 100%|██████████| 4/4 [00:01<00:00,  3.07it/s]


Batch_size after optional multiplication by num_gpus:  1


Batches: 100%|██████████| 23839/23839 [12:43<00:00, 31.22it/s]


## Debug

In [5]:
check_augmented_docs(
    augmented_docs_path=os.path.join(
        ROOT_PATH,
        "output",
        "documents_augmentation_llama3_debug",
        "docs_True_theoremqa_theorems_Llama-3.1-8B-Instruct_None_0:3.parquet"
    )
)

3
\begin{definition}[Definition:Addition]
'''Addition''' is the basic operation $+$ everyone is familiar with.
For example:
:$2 + 3 = 5$
:$47 \cdotp 3 + 191\cdotp 4 = 238 \cdotp 7$
\end{definition}

Write a final answer in the format requested.

## Step 1: Understanding the Problem

The problem is to understand the definition of addition and its application in mathematical operations.

## Step 2: Breaking Down the Definition

The definition of addition states that it is the basic operation $+$ everyone is familiar with. It provides examples of how addition is applied to numbers, such as $2 + 3 = 5$ and $47 \cdotp 3 + 191\cdotp 4 = 238 \cdotp 7$.

## Step 3: Identifying Relevant Information

The relevant information in this problem is the definition of addition and its application in mathematical operations.

## Step 4: Reasoning and Description

Addition is a fundamental operation in mathematics that combines two or more numbers to produce a sum. The definition provided shows how addit

In [48]:
augmented_docs_path = os.path.join(
    ROOT_PATH,
    "output",
    "documents_augmentation_llama3_debug",
    "theoremqa_theorems_Llama-3.1-8B-Instruct_None.parquet"
)

# Read the parquet file
augmented_docs = pq.read_table(augmented_docs_path)
augmented_docs = augmented_docs.to_pandas()

# print(augmented_docs)

doc_ids_augmented = []
documents_augmented = []
# for dp in augmented_docs:
for _, dp in augmented_docs.iterrows():
    # print(dp)
    # print(dp['doc_id'])
    # print(dp['document'])
    doc_ids_augmented.append(str(dp['doc_id']))
    documents_augmented.append(dp['document'])

In [52]:
print(len(documents_augmented))
print(documents_augmented[0])
print(doc_ids_augmented[:10])

3
\begin{definition}[Definition:Addition]
'''Addition''' is the basic operation $+$ everyone is familiar with.
For example:
:$2 + 3 = 5$
:$47 \cdotp 3 + 191\cdotp 4 = 238 \cdotp 7$
\end{definition}

What is the basic operation $+$ everyone is familiar with? 

What is the result of $2 + 3$? 

What is the result of $47 \cdotp 3 + 191\cdotp 4$? 

What is the result of $2 + 2$? 

What is the result of $3 + 4$? 

What is the result of $1 + 1$? 

What is the result of $5 + 6$? 

What is the result of $9 + 1$? 

What is the result of $7 + 8$? 

What is the result of $4 + 5$? 

What is the result of $6 + 9$? 

What is the result of $8 + 2$? 

What is the result of $10 + 3$? 

What is the result of $11 + 4$? 

What is the result of $12 + 5$? 

What is the result of $13 + 6$? 

What is the result of $14 + 7$? 

What is the result of $15 + 8$? 

What is the result of $16 + 9$? 

What is the result of $17 + 10$? 

What is the result of $18 + 11$? 

What is the result of $19 + 12$? 

What is the re

In [51]:
print(documents_augmented[1])

\begin{definition}[Definition:Addition/Complex Numbers]
The '''addition operation''' in the domain of complex numbers $\C$ is written $+$.
Let $z = a + i b, w = c + i d$ where $a, b, c, d \in \R, i^2 = -1$.
Then $z + w$ is defined as:
:$\paren {a + i b} + \paren {c + i d} = \paren {a + c} + i \paren {b + d}$
\end{definition}

What is the formula for adding two complex numbers in the domain of complex numbers $\C$? 

If $z = 3 + 4i$ and $w = 2 + 5i$, what is the value of $z + w$? 

Let $z = a + i b$ and $w = c + i d$. What is the value of $z + w$ in terms of $a, b, c, d$? 

If $z = 1 + 2i$ and $w = 3 + 4i$, what is the value of $z + w$? 

What is the value of $z + w$ if $z = 5 + 6i$ and $w = 1 + 2i$? 

Let $z = 2 + 3i$ and $w = 4 + 5i$. What is the value of $z + w$? 

If $z = 6 + 7i$ and $w = 8 + 9i$, what is the value of $z + w$? 

Let $z = 9 + 10i$ and $w = 11 + 12i$. What is the value of $z + w$? 

If $z = 12 + 13i$ and $w = 14 + 15i$, what is the value of $z + w$? 

Let $z = 15 + 16

In [50]:
print(documents_augmented[2])

\begin{definition}[Definition:Addition/Integers]
The '''addition operation''' in the domain of integers $\Z$ is written $+$.
We have that the set of integers is the Inverse Completion of Natural Numbers.
Thus it follows that elements of $\Z$ are the isomorphic images of the elements of equivalence classes of $\N \times \N$ where two tuples are equivalent if the difference between the two elements of each tuple is the same.
Thus addition can be formally defined on $\Z$ as the operation induced on those equivalence classes as specified in the definition of integers.
That is, the integers being defined as all the difference congruence classes, integer addition can be defined directly as the operation induced by natural number addition on these congruence classes:
:$\forall \tuple {a, b}, \tuple {c, d} \in \N \times \N: \eqclass {a, b} \boxminus + \eqclass {c, d} \boxminus = \eqclass {a + c, b + d} \boxminus$
\end{definition}

What is the formula used to compute the addition of two integer

## Not augmented queries

In [16]:
augmented_examples = json.load(open("/home/oh/arubinstein17/github/ReasonIR/output/documents_augmentation_llama3_debug/theoremqa_theorems_Llama-3.1-8B-Instruct_None_0:3.json"))

In [19]:
print(augmented_examples[0])
print(augmented_examples[0].keys())

{'query': '4. Review and revise the answer to ensure it is clear and concise.\n\n## Step 1: Understand the problem\nMary wants to bake 10 cookies, and each cookie can be one of three shapes: triangle, circle, and square. She wants the shapes to be as diverse as possible.\n\n## Step 2: Determine the goal\nThe goal is to find the smallest possible count for the most common shape across the ten cookies.\n\n## Step 3: Consider the constraints\nSince Mary wants the shapes to be as diverse as possible, we should aim to have an equal number of each shape, but this is not possible with 10 cookies and 3 shapes. The next best option is to have two shapes with the same number and one shape with a different number.\n\n## Step 4: Analyze the possibilities\nIf we have two shapes with 4 cookies each, that would leave 2 cookies for the third shape. This would result in a count of 4 for the two most common shapes.\n\n## Step 5: Consider alternative possibilities\nAnother option is to have one shape wit

In [21]:
queries, query_ids = get_queries(augmented_examples)

NameError: name 'get_queries' is not defined

In [ ]:
query_emb = compute_doc_embeds(
    documents=queries,
    task="theoremqa_theorems",
    save_path="/home/oh/arubinstein17/github/ReasonIR/evaluation/bright/theoremqa_theorems/augmented_query_emb.pkl",
    doc_type="queries"
)

### Debug

In [5]:
examples = load_dataset(DATASET_SOURCE, 'examples',cache_dir=CACHE_DIR)[TASK]
queries = []
query_ids = []
excluded_ids = {}
separate_reasoning = False
reasoning_examples = []
for qid, e in enumerate(examples):
    if separate_reasoning:
        new_query = f"{e['query']}\n<REASON>\n{reasoning_examples[qid]['query']}"
        queries.append(new_query)
    else:
        queries.append(e["query"])
    query_ids.append(e['id'])
    excluded_ids[e['id']] = e['excluded_ids']
    overlap = set(e['excluded_ids']).intersection(set(e['gold_ids']))
    assert len(overlap)==0
assert len(queries)==len(query_ids), f"{len(queries)}, {len(query_ids)}"

In [20]:
print(examples[0])
print(examples[0].keys())

{'query': 'Mary is planning to bake exactly 10 cookies, and each cookie may be one of three different shapes -- triangle, circle, and square. Mary wants the cookie shapes to be a diverse as possible. What is the smallest possible count for the most common shape across the ten cookies?', 'reasoning': 'Reasoning for Pigeonhole Principle: The given text provides a formal mathematical theorem and proof that directly relates to the Pigeonhole Principle, which is used to solve the given problem. The problem involves determining the minimum number of people with the same eye color in a group of 10 people, with only 3 different eye colors available. The solution correctly applies the Pigeonhole Principle by considering the eye colors as pigeonholes and the people as pigeons, leading to the conclusion that at least one color must be shared by at least 4 people.\n\nThe text provided gives a formal definition of a theorem that is essentially an expanded and rigorous form of the Pigeonhole Princip

In [13]:
query_emb = compute_doc_embeds(
    documents=queries[:DEBUG_NUMBER_EMBEDS],
    task="theoremqa_theorems",
    save_path=None,
    doc_type="queries"
)

Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00,  7.77it/s]


Batch_size after optional multiplication by num_gpus:  1


In [7]:
_, query_emb_presaved = torch.load(os.path.join(ROOT_PATH, "evaluation", "bright", 'base_emb.pkl'))

In [14]:
print(query_emb.shape)
print(query_emb_presaved.shape)
for i in range(5):
    print(query_emb[i].mean())
    print(query_emb_presaved[i].mean())
# query_emb = query_emb[0].mean()

(76, 4096)
(76, 4096)
-2.893844e-05
-2.893844e-05
1.80364e-05
1.80364e-05
-0.00032563662
-0.00032563662
0.00010392106
0.00010392106
-3.9166862e-05
-3.9166862e-05


In [15]:
assert np.isclose(query_emb, query_emb_presaved[:DEBUG_NUMBER_EMBEDS]).all()

## Not augmented docs

In [19]:
long_context = False
dataset_source = 'xlangai/BRIGHT'
task = "theoremqa_theorems"
document_postfix = ''
cache_dir = os.path.join(ROOT_PATH, "evaluation", "bright", "cache")
if long_context:
    doc_pairs = load_dataset(dataset_source, 'long_documents'+document_postfix, cache_dir=cache_dir)[task]
else:
    doc_pairs = load_dataset(dataset_source, 'documents'+document_postfix, cache_dir=cache_dir)[task]

doc_ids = []
documents = []
for dp in doc_pairs:
    doc_ids.append(dp['id'])
    documents.append(dp['content'])

In [20]:
print(len(documents))
print(documents[0])
print(doc_ids[:10])

23839
\begin{definition}[Definition:Addition]
'''Addition''' is the basic operation $+$ everyone is familiar with.
For example:
:$2 + 3 = 5$
:$47 \cdotp 3 + 191\cdotp 4 = 238 \cdotp 7$
\end{definition}
['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']


In [5]:
config_dir = os.path.join(ROOT_PATH, "evaluation", "bright", "configs")
model_arg = "reasonir"
with open(os.path.join(config_dir,model_arg.split('_ckpt')[0].split('_bilevel')[0],f"{task}.json")) as f:
    config = json.load(f)
instructions = config['instructions']

In [6]:
customized_checkpoint = 'reasonir/ReasonIR-8B'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(customized_checkpoint, torch_dtype="auto", trust_remote_code=True)
model = AutoModel.from_pretrained(customized_checkpoint, torch_dtype="auto", trust_remote_code=True)
model.eval()
model.to(device)
query_instruction = instructions['query'].format(task=task)
doc_instruction = instructions['document']
# query_max_length = kwargs.get('query_max_length',32768)
# doc_max_length = kwargs.get('doc_max_length',32768)
query_max_length = 32768
doc_max_length = 32768
print("doc max length:",doc_max_length)
print("query max length:", query_max_length)
batch_size = 1

Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00,  9.76it/s]


doc max length: 32768
query max length: 32768


In [7]:
print(doc_instruction)
print(batch_size)
print(len(documents))
print(doc_max_length)

<|embed|>

1
23839
32768


In [22]:
# Override CUDA device count to 1 to control batch processing
torch.cuda.device_count = lambda: 1
doc_emb = model.encode(documents[:DEBUG_NUMBER_EMBEDS], instruction=doc_instruction, batch_size=batch_size, max_length=doc_max_length)

In [9]:
_, doc_emb_presaved = torch.load(os.path.join(ROOT_PATH, "evaluation", "bright", 'base_doc_emb.pkl'))

In [10]:
print(doc_emb_presaved.shape)

(23839, 4096)


In [23]:
print(doc_emb_presaved[0].mean())
print(doc_emb[0].mean())
print(doc_emb_presaved[42].mean())
print(doc_emb[42].mean())
print(doc_emb_presaved[99].mean())
print(doc_emb[99].mean())

0.00029349822
0.00029349822
5.2872783e-05
5.2872783e-05
5.7571804e-05
5.7571804e-05


In [24]:
print(doc_emb.shape)
assert np.isclose(doc_emb, doc_emb_presaved[:DEBUG_NUMBER_EMBEDS]).all()

# to pass this, make sure that batch_size is 1 (by default it is multiplied by the number of GPUs)
# Override CUDA device count to 1 to control batch processing
# torch.cuda.device_count = lambda: 1

(100, 4096)
